# Séance 2 — La chasse aux erreurs

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ts/notebooks/TS_S2_chasse_aux_erreurs.ipynb)
**Objectif :** retrouver automatiquement les erreurs d’un fichier de sondages miniers, puis les corriger.

> Fichiers nécessaires : téléverser les CSV du dossier `TS_TP_donnees` (icône dossier à gauche dans Colab).
> Exécuter chaque cellule avec ▶ (ou Maj + Entrée).

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['sondages_miniers_propre.csv', 'sondages_miniers_terrain.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ts/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('sondages_miniers_terrain.csv')
print(df.shape)        # (lignes, colonnes)
df.head()

## 1. Les six questions de la check-list
Exécuter chaque ligne et **noter le résultat** dans votre cahier.

In [ ]:
print('Teneurs vides :', df['au_gpt'].isna().sum())
print('Codes -999 :', (df['au_gpt'] < 0).sum())
print('Teneurs > 50 g/t (ppb ?) :', (df['au_gpt'] > 50).sum())
print('Intervalles inversés :', (df['de_m'] >= df['a_m']).sum())
print('Lignes en double :', df.duplicated().sum())

In [ ]:
# 5. Orthographe des lithologies : combien de façons d'écrire chaque roche ?
df['lithologie'].value_counts()

In [ ]:
# Afficher les lignes suspectes
df[(df['au_gpt'] < 0) | (df['au_gpt'] > 50) | (df['de_m'] >= df['a_m'])]

## 2. Corriger
On garde toujours une copie des données brutes (`brut`).

In [ ]:
brut = df.copy()

# a) Doublons
df = df.drop_duplicates()

# b) Orthographe : enlever les espaces, majuscule au début
df['lithologie'] = df['lithologie'].str.strip().str.capitalize()
df['lithologie'] = df['lithologie'].replace({'Veine quartz': 'Veine de quartz'})

# c) Codes -999 → vide
df.loc[df['au_gpt'] < 0, 'au_gpt'] = np.nan

### À COMPLÉTER
Les teneurs supérieures à 50 g/t ont été saisies en **ppb**. Compléter la ligne pour les convertir en g/t (1 g/t = 1 000 ppb).

In [ ]:
masque = df['au_gpt'] > 50
df.loc[masque, 'au_gpt'] = df.loc[masque, 'au_gpt'] / 1000     # ← vérifier le facteur
print((df['au_gpt'] > 50).sum(), 'teneur(s) encore suspecte(s)')

### À COMPLÉTER
Pour les intervalles inversés, échanger `de_m` et `a_m`.

In [ ]:
inv = df['de_m'] > df['a_m']
df.loc[inv, ['de_m', 'a_m']] = df.loc[inv, ['a_m', 'de_m']].values
print('Intervalles inversés restants :', (df['de_m'] >= df['a_m']).sum())

## 3. Coordonnées : afficher les têtes de sondage

In [ ]:
import matplotlib.pyplot as plt
tetes = df.groupby('sondage')[['x_m', 'y_m']].first()
plt.scatter(tetes['x_m'], tetes['y_m'])
plt.xlabel('X (m)'); plt.ylabel('Y (m)'); plt.title('Têtes de sondage')
plt.show()
tetes.sort_values('x_m').tail(3)     # quel sondage est hors zone ?

## 4. Comparer avec le fichier propre

In [ ]:
propre = pd.read_csv('sondages_miniers_propre.csv')
print('Lignes :', len(df), 'vs propre :', len(propre))
print('Teneur moyenne corrigée :', round(df['au_gpt'].mean(), 3), 'g/t')
print('Teneur moyenne propre    :', round(propre['au_gpt'].mean(), 3), 'g/t')
print('Teneur moyenne brute     :', round(brut['au_gpt'].mean(), 3), 'g/t  ← effet des erreurs !')

## Bilan (à écrire)
1. Combien d’erreurs de chaque type avez-vous trouvées ?
2. Laquelle avait le plus d’effet sur la teneur moyenne ?
3. Lesquelles auriez-vous vues à l’œil dans Excel ?